# GST on NTU RGB+D 60 (Kaggle)
This notebook clones the public project repository; enable *Internet* in the notebook settings.

In [ ]:
REPO = "Muhammad-Huzifa/gst-ntu-skeleton-action-recognition"
!rm -rf /kaggle/working/gst_repo
!git clone -q https://github.com/{REPO}.git /kaggle/working/gst_repo
!cd /kaggle/working/gst_repo && git log --oneline | head -3
%cd /kaggle/working/gst_repo

## 1. Preprocess (3D, 2 persons, spine-centred + view-aligned) → one packed array
Re-run only if `/kaggle/working/ntu60_packed` is missing. Save it as a Kaggle Dataset afterwards to skip this step next time (~3.4 GB).

In [ ]:
!python -m gst.data.ntu_preprocess --input /kaggle/input/datasets/huzifaulhaq/ntu-60/nturgb+d_skeletons --out /kaggle/working/ntu60_packed --workers 4

## 2. Quick sanity run (2 epochs on 2000 samples) before the long run

In [ ]:
!python train_ntu.py --data /kaggle/working/ntu60_packed --out /kaggle/working/outputs/smoke --epochs 2 --max-train 2000 --eval-every 1

## 3. Train GST (cross-subject). Re-run this cell with `--resume` if the session dies; checkpoints are in /kaggle/working/outputs/gst

In [ ]:
!python train_ntu.py --data /kaggle/working/ntu60_packed --out /kaggle/working/outputs/gst --arch gst --epochs 120 --resume

## 4. Baseline on the SAME pipeline (your old architecture), needed to attribute any gain to the architecture rather than the data fixes

In [ ]:
!python train_ntu.py --data /kaggle/working/ntu60_packed --out /kaggle/working/outputs/baseline --arch baseline --epochs 120 --resume

In [ ]:
import json
for n in ["gst", "baseline"]:
    try: print(n, json.load(open(f"/kaggle/working/outputs/{n}/final.json"))["final"])
    except FileNotFoundError: print(n, "not finished")